# 🏥 Dermato-RAG: Faz 4 - Vision Model Fine-Tuning (Colab - ZIP Versiyonu)

Bu notebook, bilgisayarınızda sıkıştırıp Google Drive'a yüklediğiniz **`Dermato-RAG.zip`** dosyasını hızlıca açarak BiomedCLIP eğitimini başlatır.

### 1. Google Drive'ı Bağla
Drive'a erişim izni verin ki yüklediğiniz ZIP dosyasını çekebilelim.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Projeyi ZIP'ten Çıkar (Hızlı Metot)
Drive'ınızdaki Dermato-RAG.zip dosyasını Colab'ın yerel süper hızlı diskine kopyalayıp açıyoruz. (Drive'dan okumak yavaştır, bu yüzden buraya kopyalıyoruz).

In [ ]:
import os

# Eğer klasör zaten varsa silip temizleyelim
!rm -rf /content/Dermato-RAG

# ZIP dosyasını aç (Drive'da ana dizine attığını varsayıyoruz)
print("ZIP dosyası açılıyor, lütfen bekleyin (1-2 dakika sürebilir)...")
!unzip -q /content/drive/MyDrive/Dermato-RAG.zip -d /content/
print("Dosyalar başarıyla açıldı!")

# Çalışma dizinini projenin içine alıyoruz
os.chdir('/content/Dermato-RAG')

# Gerekli kütüphaneleri kuruyoruz
!pip install -r requirements.txt --quiet
!pip install open_clip_torch --quiet

### 3. Eğitim Kurulumu ve Veri Yükleme

In [ ]:
import sys
sys.path.append('/content/Dermato-RAG')

import torch
import os
from src.data.dataset import get_dataloaders
from src.models.vision_encoder import DermatoVisionEncoder
from src.models.trainer import VisionTrainer

# Modellerin Drive'a kaydolması için çıktı klasörü oluştur
output_dir = "/content/drive/MyDrive/Dermato-RAG_Checkpoints"
os.makedirs(output_dir, exist_ok=True)

# DataLoader'ları oluştur
train_loader, val_loader, test_loader = get_dataloaders(
    batch_size=32,
    image_size=224,
    num_workers=2,
    use_weighted_sampler=True
)

### 4. Modeli Başlat (Sadece Head Eğitimi)

In [ ]:
model = DermatoVisionEncoder(
    num_classes=9, 
    mode="classify",
    pretrained=True,
    freeze_backbone=True
)

trainer = VisionTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    output_dir=output_dir
)

history = trainer.train(num_epochs=5, early_stopping_patience=3, use_amp=True)

### 5. Tüm Modeli İnce Ayar Yapma (Fine-Tuning)

In [ ]:
model.unfreeze_backbone(unfreeze_last_n=12)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5, 
    weight_decay=0.01
)

trainer.optimizer = optimizer
trainer.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

history_finetune = trainer.train(num_epochs=10, early_stopping_patience=4, use_amp=True)

### 6. Test Setinde Değerlendirme

In [ ]:
best_model_path = os.path.join(output_dir, "best_model.pt")
model = DermatoVisionEncoder.load_checkpoint(best_model_path, device="cuda")

trainer.model = model
test_results = trainer.evaluate(test_loader)
print("\nTest Sonuçları:", test_results)